# MoeLoRA Llama Demo
This notebook shows how to load a pretrained Llama model via `transformers`, wrap it with PEFT's MoeLoRA, and run a quick forward pass.

In [ ]:
from transformers import AutoModelForCausalLM
from model.peft import MoeLoraConfig, get_peft_model
from model.peft.tuners.gating import GATING_TO_MODEL_MAPPING
import torch

In [ ]:
model_path = 'path/to/llama'
base_model = AutoModelForCausalLM.from_pretrained(model_path)

In [ ]:
moelora_cfg = MoeLoraConfig(
    task_type='CAUSAL_LM',
    inference_mode=False,
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=None,
    num_moe=2,
    gating='Dense',
)
model = get_peft_model(base_model, moelora_cfg)
model.print_trainable_parameters()

In [ ]:
gate_model = GATING_TO_MODEL_MAPPING[moelora_cfg.gating](dim=model.config.hidden_size, num_moe=moelora_cfg.num_moe)

input_ids = torch.randint(0, model.config.vocab_size, (2, 5))
user_embeds = torch.randn(2, gate_model.dim)
with torch.no_grad():
    gate_weights = gate_model(user_embeds).unsqueeze(1)
    out = model(input_ids=input_ids, user_embeds=user_embeds, gate_weights=gate_weights)
print(out.logits.shape)